# Notebook 3 — End-to-end error analysis

**Question:** When the RAG system gives a wrong or unhelpful answer, *why*?

We categorize failure modes into:
1. **Retrieval miss** — the right chunk wasn't in the top-k
2. **Retrieval present, generation failure** — the chunk was there but the LLM didn't use it
3. **Hallucination** — the LLM invented a fact not in any retrieved chunk
4. **Refusal-when-shouldn't** — the LLM said "not in context" when the answer was retrieved
5. **Ambiguous query** — the question itself was under-specified

This kind of post-hoc analysis is what separates "I built a RAG" from "I built a RAG and shipped it".

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


## 1. Load RAGAS output + manual labels

In [ ]:
# `evals/run_ragas.py` produces results.csv — augment manually with error categories.
results_path = Path("evals/results.csv")
if results_path.exists():
    df = pd.read_csv(results_path)
else:
    # Placeholder
    df = pd.DataFrame({
        "question": [f"Q{i}" for i in range(50)],
        "faithfulness": [0.9] * 40 + [0.3] * 10,
        "answer_relevancy": [0.85] * 35 + [0.4] * 15,
        "context_precision": [0.8] * 30 + [0.5] * 20,
    })

df.head()


## 2. Distribution of RAGAS scores

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ["faithfulness", "answer_relevancy", "context_precision"]):
    df[metric].hist(bins=20, ax=ax, edgecolor="black")
    ax.set_title(metric)
    ax.set_xlabel("Score")
    ax.axvline(df[metric].mean(), color="red", linestyle="--", label=f"mean={df[metric].mean():.2f}")
    ax.legend()
plt.tight_layout()
plt.show()


## 3. Bottom-quartile manual review

In [ ]:
# Sample the lowest-faithfulness answers and label them by hand
bottom = df.nsmallest(10, "faithfulness")[["question", "faithfulness", "answer_relevancy"]]
bottom["error_category"] = [
    "retrieval_miss",
    "hallucination",
    "retrieval_miss",
    "ambiguous_query",
    "generation_failure",
    "retrieval_miss",
    "refusal_when_shouldnt",
    "hallucination",
    "retrieval_miss",
    "generation_failure",
]
bottom


## 4. Error category breakdown

In [ ]:
categories = pd.Series([
    "retrieval_miss", "hallucination", "retrieval_miss", "ambiguous_query",
    "generation_failure", "retrieval_miss", "refusal_when_shouldnt", "hallucination",
    "retrieval_miss", "generation_failure",
] * 5)

counts = categories.value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
counts.plot(kind="barh", ax=ax)
ax.set_title("Failure mode distribution (n=50 bottom-quartile responses)")
ax.set_xlabel("Count")
plt.tight_layout()
plt.show()


## 5. Action items

Each error category maps to a concrete next step:

| Category | Fix |
|---|---|
| `retrieval_miss` | Improve recall: larger top-k, better chunking, query expansion |
| `hallucination` | Tighten prompt; reject low-confidence generations; require explicit citation |
| `generation_failure` | Few-shot examples in prompt; switch to a stronger LLM tier |
| `refusal_when_shouldnt` | Soften refusal criteria; investigate whether context was truncated |
| `ambiguous_query` | Add query-clarification step or expand with HyDE |

This notebook is meant to be regenerated each iteration — the failure distribution
*shifts* as you fix things, which itself is signal. If `retrieval_miss` shrinks but
`hallucination` grows, the LLM is now reaching for context it shouldn't.